# Descriptive Statistics: HR & HRV

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../')  # go up to scripts/ root


import pandas as pd
import numpy as np
from scipy import stats

import matplotlib.pyplot as plt

from datetime import datetime

from plot_helper.descriptive_stats_plot import plot_descriptive_stats
from plot_helper.colors import COLORS

# Functions

In [ ]:
def check_normality(data, feature_name="Feature", ax=None):
    """
    Generate Q-Q plot and run Shapiro-Wilk test for a given array/series.
    
    Parameters
    ----------
    data : array-like
        Numeric data to test (NaNs are dropped automatically).
    feature_name : str
        Label used in the plot title and printed output.
    ax : matplotlib axis, optional
        If provided, plots on this axis (useful for subplots/loops).
        Otherwise creates its own figure.
    
    Returns
    -------
    dict with W statistic, p-value, and a boolean for normality at alpha=0.05
    """
    data = pd.Series(data).dropna().values

    # Shapiro-Wilk test
    W, p_value = stats.shapiro(data)
    is_normal = p_value > 0.05

    # Q-Q plot
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 5))
    stats.probplot(data, dist="norm", plot=ax)
    ax.set_title(f"Q-Q Plot: {feature_name}\nShapiro-Wilk W={W:.3f}, p={p_value:.4f}")

    print(f"{feature_name}: W={W:.4f}, p={p_value:.4f} -> "
          f"{'Normal (fail to reject H0)' if is_normal else 'Non-normal (reject H0)'}")

    return {"feature": feature_name, "W": W, "p_value": p_value, "is_normal": is_normal}



# --- Example: loop over multiple features in a feature matrix ---
def check_normality_batch(df, feature_cols, ncols=4):
    n = len(feature_cols)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
    axes = np.array(axes).reshape(-1)

    results = []
    for i, col in enumerate(feature_cols):
        data = df[col].dropna()
        W, p_value = stats.shapiro(data)
        skewness = stats.skew(data, bias=False)

        if abs(skewness) < 0.5:
            direction = "symmetric"
        elif skewness >= 0.5:
            direction = "right-skewed"
        else:
            direction = "left-skewed"

        stats.probplot(data, dist="norm", plot=axes[i])
        axes[i].set_title(f"{col}\nW={W:.3f}, p={p_value:.4f}\nskew={skewness:.2f} ({direction})")

        results.append({
            "feature": col, "W": W, "p_value": p_value,
            "is_normal": p_value > 0.05,
            "skewness": skewness, "direction": direction
        })

    for j in range(len(feature_cols), len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()
    return pd.DataFrame(results)

## Get calculated HR/HRV (R)

In [ ]:
df_daily_hr = pd.read_csv("../../output/1_feature_extraction/df_features_daily_hr_2026-07-08.csv")
df_daily_nocturnal_hr = pd.read_csv("../../output/1_feature_extraction/df_features_daily_nocturnal_hr_2026-07-08.csv")
df_hr = pd.read_csv("../../output/1_feature_extraction/df_features_hr_2026-07-08.csv")
df_nocturnal_hr = pd.read_csv("../../output/1_feature_extraction/df_features_nocturnal_hr_2026-07-08.csv")

print("Daily HR:")
print(df_daily_hr.shape)
display(df_daily_hr.head())
print(len(df_daily_hr['study_id'].unique()))
print("Daily Nocturnal HR:")
print(df_daily_nocturnal_hr.shape)
display(df_daily_nocturnal_hr.head())
print(len(df_daily_nocturnal_hr['study_id'].unique()))
print("HR:")
print(df_hr.shape)
display(df_hr.head())
print(len(df_hr['study_id'].unique()))
print("Nocturnal HR:")
print(df_nocturnal_hr.shape)
display(df_nocturnal_hr.head())
print(len(df_nocturnal_hr['study_id'].unique()))

#check which patients have less than 7 days/nights of data
df_hr_exclude = df_hr[df_hr['n_days'] < 7]
print("Patients with less than 7 days of data:")
display(df_hr_exclude)
print(len(df_hr_exclude['study_id'].unique()))
df_nocturnal_hr_exclude = df_nocturnal_hr[df_nocturnal_hr['n_nights'] < 7]
print("Patients with less than 7 nights of data:")
display(df_nocturnal_hr_exclude)
print(len(df_nocturnal_hr_exclude['study_id'].unique()))

#exclude patients with less than 7 days/nights of data
df_hr = df_hr[df_hr['n_days'] >= 7]
df_nocturnal_hr = df_nocturnal_hr[df_nocturnal_hr['n_nights'] >= 7]

display(df_hr.shape)
display(df_nocturnal_hr.shape)

## HR

In [ ]:
df_daily_hr = df_daily_hr.sort_values(by=['study_id']).reset_index(drop=True)
df_hr = df_hr.sort_values(by=['study_id']).reset_index(drop=True)
n = len(df_daily_hr['study_id'].unique())
#create plots
fig_f1 = plot_descriptive_stats(
    df = df_daily_hr,
    col_df = "daily_mean_hr", 
    df_descriptive_stats=df_hr,
    col_mean="mean_hr",
    col_median="median_hr", 
    n_patients=n,
    feature="HR",
    title="HR",
    tickformat=".0f",
    colors = COLORS
    )

fig_f1.show()

## HRV

In [ ]:
#create plots
fig_f2 = plot_descriptive_stats(
    df = df_daily_hr,
    col_df = "daily_mean_rmssd", 
    df_descriptive_stats=df_hr,
    col_mean="mean_rmssd",
    col_median="median_rmssd", 
    n_patients=n,
    feature="HRV (RMSSD)",
    title="HRV (RMSSD)",
    tickformat=".0f",
    colors = COLORS
    )

fig_f2.show()

## Nocturnal HR

In [ ]:
df_daily_nocturnal_hr = df_daily_nocturnal_hr.sort_values(by=['study_id']).reset_index(drop=True)
df_nocturnal_hr = df_nocturnal_hr.sort_values(by=['study_id']).reset_index(drop=True)
n = len(df_daily_nocturnal_hr['study_id'].unique())
#create plots
fig_f3 = plot_descriptive_stats(
    df = df_daily_nocturnal_hr,
    col_df = "daily_mean_hr", 
    df_descriptive_stats=df_nocturnal_hr,
    col_mean="mean_hr",
    col_median="median_hr", 
    n_patients=n,
    feature="Nocturnal HR",
    title="Nocturnal HR",
    tickformat=".0f",
    colors = COLORS
    )

fig_f3.show()

## Nocturnal HRV


In [ ]:
#create plots
fig_f4 = plot_descriptive_stats(
    df = df_daily_nocturnal_hr,
    col_df = "daily_mean_rmssd", 
    df_descriptive_stats=df_nocturnal_hr,
    col_mean="mean_rmssd",
    col_median="median_rmssd", 
    n_patients=n,
    feature="Nocturnal HRV (RMSSD)",
    title="Nocturnal HRV (RMSSD)",
    tickformat=".0f",
    colors = COLORS
    )

fig_f4.show()

## Calculate descriptive statistics

In [ ]:
df_stats = pd.merge(df_hr[['study_id', 'mean_hr', 'mean_rmssd']], df_nocturnal_hr[['study_id', 'mean_hr', 'mean_rmssd']], on='study_id', how='outer', suffixes=('', '_nocturnal_hr'))
display(df_stats.columns)

#calculate mean, median, standard deviation, min, max
df_stats = df_stats.describe().transpose().reset_index()
display(df_stats)
date = datetime.now().strftime("%Y-%m-%d")
df_stats.to_csv(f'../../output/2_descriptive_stats/df_features_hr_stats_{date}.csv', index=False)

### Q-Q Plot & Shapiro-Wilk Test

In [ ]:
hr_feature_cols = ["mean_hr", "mean_rmssd"]
hr_summary = check_normality_batch(df_hr, hr_feature_cols, ncols=2)
print(hr_summary, f"\n")
date = datetime.now().strftime("%Y-%m-%d")
hr_summary.to_csv(f"../../output/2_descriptive_stats/hr_normality_summary_{date}.csv", index=False)

nocturnal_hr_feature_cols = ["mean_hr", "mean_rmssd"]
nocturnal_hr_summary = check_normality_batch(df_nocturnal_hr, nocturnal_hr_feature_cols, ncols=2)
print(nocturnal_hr_summary, f"\n")
nocturnal_hr_summary.to_csv(f"../../output/2_descriptive_stats/nocturnal_hr_normality_summary_{date}.csv", index=False)